# Atelier Scikit-learn
Contexte
Une entreprise possède plusieurs bâtiments équipés de capteurs IoT.
Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression,
la consommation énergétique, le bâtiment, la date et l'heure de la mesure.
Chaque mesure possède également un état (OK, ALERTE et ERREUR).
L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un
capteur à partir de ses mesures.
L'atelier suivra le workflow classique du Machine Learning :
Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement →
Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation

## Partie 0 — Mise en place de l'environnement

**Objectif :** préparer l'environnement de travail et charger les données.

In [1]:
# pip install seaborn matplotlib pandas scikit-learn  (si nécessaire)

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

sns.set_theme()
%matplotlib inline

MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

### Import du dataset

In [3]:
df = pd.read_csv("../data/mesures_capteurs.csv", parse_dates=["date_heure"])
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


### Exploration du DataFrame

In [7]:
print("Dimensions du data fram :", df.shape)
print("L'information du datafram ")
df.info()

Dimensions du data fram : (605, 9)
L'information du datafram 
<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_mesure     605 non-null    str           
 1   date_heure    605 non-null    datetime64[us]
 2   id_capteur    605 non-null    str           
 3   batiment      605 non-null    str           
 4   temperature   599 non-null    float64       
 5   humidite      600 non-null    float64       
 6   pression      600 non-null    float64       
 7   consommation  600 non-null    float64       
 8   etat          601 non-null    str           
dtypes: datetime64[us](1), float64(4), str(4)
memory usage: 42.7 KB


In [8]:
df.describe()

,date_heure,temperature,humidite,pression,consommation
count,605,599.000000,600.00000,600.000000,600.000000
mean,2026-01-17 11:32:43.636363,24.878314,64.92620,1012.221900,208.675417
min,2026-01-05 00:00:00,-18.500000,28.52000,850.000000,18.120000
25%,2026-01-11 05:00:00,22.570000,58.17250,1006.790000,160.177500
50%,2026-01-17 12:00:00,24.860000,65.37500,1012.855000,206.150000
75%,2026-01-23 18:00:00,27.275000,71.61500,1017.827500,254.127500
max,2026-01-29 23:00:00,58.700000,145.00000,1038.430000,875.000000
std,NaN,4.059576,10.76905,10.599042,72.243567


In [9]:
df["etat"].value_counts()

etat
OK        567
ALERTE     29
ERREUR      5
Name: count, dtype: int64

**Constat :** le dataset contient 605 mesures. La variable cible `etat` est **fortement
déséquilibrée** : très majoritairement `OK` (567 mesures), avec seulement 29 `ALERTE` et 5 `ERREUR`.


## Partie 1 — Gestion des doublons

**Objectif :** un doublon (ligne strictement identique à une autre) fausserait l'entraînement en
donnant artificiellement plus de poids à certaines observations — il faut donc les détecter et les
supprimer avant toute étape de modélisation.

### 1) Vérifier l'existence de doublons

In [12]:
# .duplicated() renvoie True pour chaque ligne identique à une ligne précédente
# .sum() compte le nombre total de doublons détectés
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons détectés : {nb_doublons}")

Nombre de doublons détectés : 5


### 2) Supprimer les doublons puis vérifier

In [13]:
df = df.drop_duplicates()

print("Dimensions après suppression :", df.shape)
print("Doublons restants :", df.duplicated().sum())

Dimensions après suppression : (600, 9)
Doublons restants : 0
